In [ ]:
# ── Section 0: Install wheels + diagnose mount paths ─────────────────────────
import subprocess, sys, os

# Show what's actually mounted so we can debug path issues
print('=== /kaggle/input contents ===')
try:
    for name in sorted(os.listdir('/kaggle/input')):
        print(f'  {name}')
except Exception as e:
    print(f'  (cannot list: {e})')

def find_path(*candidates):
    for c in candidates:
        if os.path.exists(c):
            print(f'  Found: {c}')
            return c
    print(f'  Not found. Tried: {candidates}')
    return None

def install_wheel(path, name=''):
    if not path or not os.path.exists(path):
        print(f'  WARNING: wheel not found: {path}')
        return False
    print(f'  Installing {name or os.path.basename(path)}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', path],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  STDERR: {result.stderr[-300:]}')
        return False
    print('  OK')
    return True

# Kaggle datasets mount at /kaggle/input/<slug>/  (slug = part after username)
# Fallback to /kaggle/input/datasets/<user>/<slug>/ for older kernel API
print('\nLocating wheels...')
BIOPYTHON_WHL = find_path(
    '/kaggle/input/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl',
    '/kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl',
)
BIOTITE_WHL = find_path(
    '/kaggle/input/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl',
    '/kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl',
)
RDKIT_WHL = find_path(
    '/kaggle/input/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl',
    '/kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl',
)

print('\nInstalling wheels...')
install_wheel(BIOPYTHON_WHL, 'biopython')
install_wheel(BIOTITE_WHL,   'biotite')
install_wheel(RDKIT_WHL,     'rdkit')
print('Wheel installs done.')

In [ ]:
# ── Section 1: GPU and Protenix setup ────────────────────────────────────────
import os, sys, torch

os.environ['LAYERNORM_TYPE']          = 'torch'
os.environ['RNA_MSA_DEPTH_LIMIT']     = '512'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# Try both Kaggle dataset mount patterns
PROTENIX_ROOT = None
for _root in [
    '/kaggle/input/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1',
    '/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1',
    '/kaggle/input/protenix-v1-adjusted',
]:
    if os.path.exists(_root):
        PROTENIX_ROOT = _root
        break

PROTENIX_AVAILABLE = PROTENIX_ROOT is not None
if PROTENIX_AVAILABLE:
    os.environ['PROTENIX_ROOT_DIR'] = PROTENIX_ROOT
    if PROTENIX_ROOT not in sys.path:
        sys.path.insert(0, PROTENIX_ROOT)
    CHECKPOINT_PATH = os.path.join(PROTENIX_ROOT, 'checkpoint',
                                   'protenix_base_20250630_v1.0.0.pt')
    PROTENIX_AVAILABLE = os.path.exists(CHECKPOINT_PATH)
    print(f'Protenix root      : {PROTENIX_ROOT}')
    print(f'Checkpoint exists  : {PROTENIX_AVAILABLE}')
else:
    CHECKPOINT_PATH = ''
    print('WARNING: Protenix not found — will use TBM-only (no regression from baseline)')

MODEL_NAME = 'protenix_base_20250630_v1.0.0'  # exact key in configs_model_type.py

DATA_PATH = None
for _candidate in [
    '/kaggle/input/competitions/stanford-rna-3d-folding-2',
    '/kaggle/input/stanford-rna-3d-folding-2',
]:
    if os.path.exists(_candidate):
        DATA_PATH = _candidate
        break
if DATA_PATH is None:
    raise RuntimeError('Competition data not found')

OUT_PATH = '/kaggle/working'
os.makedirs(OUT_PATH, exist_ok=True)

print(f'GPU               : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}  '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'Data path         : {DATA_PATH}')
print(f'Protenix available: {PROTENIX_AVAILABLE}')

In [ ]:
# ── Section 2: TBM functions ─────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation
from scipy.spatial.distance import cdist
import time


def needleman_wunsch(seq_a, seq_b, match=2, mismatch=-1, gap_open=-10, gap_extend=-0.5):
    n, m = len(seq_a), len(seq_b)
    NEG_INF = float('-inf')
    M = np.full((n+1, m+1), NEG_INF)
    X = np.full((n+1, m+1), NEG_INF)
    Y = np.full((n+1, m+1), NEG_INF)
    M[0, 0] = 0.0
    for i in range(1, n+1):
        X[i, 0] = gap_open + (i-1) * gap_extend
    for j in range(1, m+1):
        Y[0, j] = gap_open + (j-1) * gap_extend
    for i in range(1, n+1):
        for j in range(1, m+1):
            s = match if seq_a[i-1] == seq_b[j-1] else mismatch
            M[i,j] = max(M[i-1,j-1], X[i-1,j-1], Y[i-1,j-1]) + s
            X[i,j] = max(M[i-1,j] + gap_open, X[i-1,j] + gap_extend)
            Y[i,j] = max(M[i,j-1] + gap_open, Y[i,j-1] + gap_extend)
    score = max(M[n,m], X[n,m], Y[n,m])
    a_aln, b_aln = [], []
    i, j = n, m
    state = np.argmax([M[n,m], X[n,m], Y[n,m]])
    while i > 0 or j > 0:
        if state == 0:
            a_aln.append(seq_a[i-1]); b_aln.append(seq_b[j-1])
            prev = np.argmax([M[i-1,j-1], X[i-1,j-1], Y[i-1,j-1]])
            i -= 1; j -= 1; state = prev
        elif state == 1:
            a_aln.append(seq_a[i-1]); b_aln.append('-')
            state = 0 if M[i-1,j] + gap_open >= X[i-1,j] + gap_extend else 1
            i -= 1
        else:
            a_aln.append('-'); b_aln.append(seq_b[j-1])
            state = 0 if M[i,j-1] + gap_open >= Y[i,j-1] + gap_extend else 2
            j -= 1
    return ''.join(reversed(a_aln)), ''.join(reversed(b_aln)), score


class Config:
    MAX_RELATIVE_LENGTH_DIFF  = 0.5
    MAX_ALIGN_LENGTH          = 500
    ALIGNMENT_MATCH           = 2
    ALIGNMENT_MISMATCH        = -1
    ALIGNMENT_GAP_OPEN        = -10   # heavy gap penalty → prefers matches
    ALIGNMENT_GAP_EXTEND      = -0.5
    BOND_DISTANCE_TARGET      = 6.0   # C1'-C1' spacing in RNA (Å)
    BOND_DISTANCE_TOL         = 0.5
    MIN_NONBOND_DISTANCE      = 3.8   # v8: stronger clash removal
    BASE_PAIRING_DISTANCE_IDEAL = 12.5  # ideal WC C1'–C1' distance (Å)
    BASE_PAIRING_DISTANCE_RANGE = (8.0, 14.0)
    HELIX_RADIUS              = 10.0
    HELIX_RISE_PER_BASE       = 2.5
    HELIX_ANGLE_STEP          = 0.6
    PAIRING_PROB_THRESHOLD    = 0.7
    STEP_LENGTH_RANGE         = (3.5, 4.5)
    NUM_PREDICTIONS           = 5
    DEFAULT_RELIABILITY       = 0.2
    NOISE_SCALE_MIN           = 0.01
    RANDOM_SEED_OFFSET        = 1000
    TBM_ROUTING_SIM_THRESHOLD = 0.55   # send to Protenix if best_sim below this


def extract_structures(labels_df):
    df = labels_df.copy()
    df['target_id'] = df['ID'].str.rsplit('_', n=1).str[0]
    structs = {}
    for target_id, group in df.groupby('target_id', sort=False):
        coords = group.sort_values('resid')[['x_1', 'y_1', 'z_1']].values.astype(np.float32)
        structs[target_id] = coords
    return structs


def secondary_structure_filter(seq_a, seq_b, min_bp=2):
    """Quick check: do seq_a and seq_b have complementary bases at ≥min_bp positions?"""
    pairing = {'A': 'U', 'U': 'A', 'G': 'C', 'C': 'G'}
    count = sum(
        1 for i in range(min(len(seq_a), len(seq_b)))
        if seq_a[i] in pairing and seq_b[i] == pairing[seq_a[i]]
    )
    return count >= min_bp


def compute_alignment(query_seq, template_seq):
    a_aln, b_aln, raw_score = needleman_wunsch(
        query_seq, template_seq,
        match=Config.ALIGNMENT_MATCH, mismatch=Config.ALIGNMENT_MISMATCH,
        gap_open=Config.ALIGNMENT_GAP_OPEN, gap_extend=Config.ALIGNMENT_GAP_EXTEND,
    )
    norm_score = raw_score / (2 * min(len(query_seq), len(template_seq)))
    return (a_aln, b_aln), norm_score


def find_templates(query_seq, template_df, struct_dict, date_cutoff=None, top_k=5):
    """
    Return (top_k templates, best_sim). Long sequences return ([], 0.0) for Protenix.
    """
    if len(query_seq) > Config.MAX_ALIGN_LENGTH:
        return [], 0.0
    candidates = (template_df[template_df['temporal_cutoff'] < date_cutoff]
                  if date_cutoff else template_df)
    matches  = []
    best_sim = 0.0
    for _, row in candidates.iterrows():
        tid  = row['target_id']
        tseq = row['sequence']
        if tid not in struct_dict:
            continue
        rel = abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq))
        if rel > Config.MAX_RELATIVE_LENGTH_DIFF:
            continue
        # Pre-filter: templates with no WC complementarity are poor RNA models
        if not secondary_structure_filter(query_seq, tseq):
            continue
        (a_aln, b_aln), score = compute_alignment(query_seq, tseq)
        best_sim = max(best_sim, score)
        matches.append((tid, tseq, score, struct_dict[tid], a_aln, b_aln))
    matches.sort(key=lambda x: x[2], reverse=True)
    return matches[:top_k], best_sim


def morph_template(dest_seq, aligned_dest, aligned_source, source_coords):
    morphed = np.full((len(dest_seq), 3), np.nan, dtype=np.float32)
    d_i = s_i = 0
    for a, b in zip(aligned_dest, aligned_source):
        if a != '-' and b != '-':
            if s_i < len(source_coords):
                morphed[d_i] = source_coords[s_i]
            d_i += 1; s_i += 1
        elif a != '-':
            d_i += 1
        else:
            s_i += 1
    valid = ~np.isnan(morphed[:, 0])
    if not valid.any():
        return make_de_novo_structure(dest_seq)
    idx = np.arange(len(morphed))
    for dim in range(3):
        morphed[:, dim] = np.interp(
            idx, idx[valid], morphed[valid, dim],
            left=morphed[valid, dim][0], right=morphed[valid, dim][-1]
        )
    return morphed


def refine_geometry(positions, sequence, reliability=1.0):
    """
    RNA-aware geometry refinement (v8 approach restored):
    - Bond length correction (~6 Å consecutive C1')
    - Angle smoothing (no kinks > 143°)
    - Steric clash removal (sampled for O(n) scaling)
    - Watson-Crick base pair distance nudging (key for RNA quality)
    """
    refined  = positions.copy().astype(np.float32)
    n        = len(sequence)
    # v8 strength formula: low reliability → push harder, high reliability → gentle
    strength = 0.8 * (1.0 - min(reliability, 0.8))

    # Bond length correction (consecutive C1' atoms ~6 Å apart)
    for i in range(n - 1):
        vec  = refined[i+1] - refined[i]
        dist = np.linalg.norm(vec)
        if abs(dist - Config.BOND_DISTANCE_TARGET) > Config.BOND_DISTANCE_TOL:
            unit  = vec / (dist + 1e-10)
            delta = (Config.BOND_DISTANCE_TARGET - dist) * strength
            refined[i+1] = refined[i] + unit * (dist + delta)

    # Angle smoothing: avoid hairpin kinks > ~143°
    if n > 3:
        for i in range(1, n - 1):
            prev  = refined[i] - refined[i-1]
            nxt   = refined[i+1] - refined[i]
            denom = np.linalg.norm(prev) * np.linalg.norm(nxt) + 1e-10
            angle = np.arccos(np.clip(np.dot(prev, nxt) / denom, -1, 1))
            if angle > 2.5:  # ~143°
                smoothed   = (refined[i-1] + refined[i+1]) / 2
                refined[i] = refined[i] * 0.3 + smoothed * 0.7

    # Steric clash removal (sampled subset for efficiency on long sequences)
    if n >= 10:
        sub_idx = np.linspace(0, n-1, min(n, 200), dtype=int)
        sub     = refined[sub_idx]
        dmat    = cdist(sub, sub)
        np.fill_diagonal(dmat, np.inf)
        ii, jj = np.where((dmat < Config.MIN_NONBOND_DISTANCE) &
                          (np.abs(sub_idx[:, None] - sub_idx[None, :]) > 2))
        for k in range(len(ii)):
            gi, gj = sub_idx[ii[k]], sub_idx[jj[k]]
            vec  = refined[gj] - refined[gi]
            dist = np.linalg.norm(vec) + 1e-10
            push = (Config.MIN_NONBOND_DISTANCE - dist) * strength
            refined[gi] -= (vec / dist) * push * 0.5
            refined[gj] += (vec / dist) * push * 0.5

    # Watson-Crick base-pair distance nudging (key RNA constraint)
    wc_pairs = {'A': 'U', 'U': 'A', 'G': 'C', 'C': 'G'}
    for i in range(n):
        partner = wc_pairs.get(sequence[i])
        if not partner:
            continue
        for j in range(i + 3, min(i + 20, n)):
            if sequence[j] == partner:
                d  = np.linalg.norm(refined[i] - refined[j])
                lo, hi = Config.BASE_PAIRING_DISTANCE_RANGE
                if lo < d < hi:
                    delta = (Config.BASE_PAIRING_DISTANCE_IDEAL - d) * strength * 0.3
                    unit  = (refined[j] - refined[i]) / (d + 1e-10)
                    refined[i] -= unit * delta * 0.5
                    refined[j] += unit * delta * 0.5
                break

    return refined


def make_de_novo_structure(seq, seed=None, rng=None):
    """
    Build a rough 3D RNA structure from scratch using idealised A-form helix
    parameters with probabilistic Watson-Crick base-pairing placement.
    """
    if rng is None:
        rng = np.random.default_rng(seed)
    n      = len(seq)
    coords = np.zeros((n, 3), dtype=np.float32)
    pairs  = {'G': 'C', 'C': 'G', 'A': 'U', 'U': 'A'}

    if n <= 3:
        for i in range(n):
            coords[i] = [i * 6.0, 0.0, 0.0]
        return coords

    # Seed first few residues in a helix
    start = min(4, n // 2)
    for i in range(start):
        angle     = i * Config.HELIX_ANGLE_STEP * 0.8
        coords[i] = [
            Config.HELIX_RADIUS * 0.7 * np.cos(angle),
            Config.HELIX_RADIUS * 0.7 * np.sin(angle),
            i * Config.HELIX_RISE_PER_BASE * 1.2
        ]

    direction = np.array([0.0, 0.1, 0.995])
    direction /= np.linalg.norm(direction)

    for i in range(start, n):
        found_pair = False
        for j in range(max(0, i - 12), i):
            if seq[j] == pairs.get(seq[i]) and (i - j) <= 8:
                if rng.random() < Config.PAIRING_PROB_THRESHOLD:
                    vec = coords[i-1] - coords[max(0, i-2)]
                    if np.linalg.norm(vec) > 1e-6:
                        perp = np.cross(vec, np.array([0, 0, 1]))
                        perp /= np.linalg.norm(perp) + 1e-10
                        coords[i] = coords[i-1] + perp * (Config.BASE_PAIRING_DISTANCE_IDEAL * 0.6)
                    else:
                        r = rng.standard_normal(3)
                        coords[i] = coords[i-1] + r / (np.linalg.norm(r) + 1e-10) * Config.BASE_PAIRING_DISTANCE_IDEAL * 0.6
                    direction = rng.standard_normal(3)
                    direction /= np.linalg.norm(direction) + 1e-10
                    found_pair = True
                    break
        if not found_pair:
            if rng.random() < 0.25:
                axis = rng.standard_normal(3); axis /= np.linalg.norm(axis) + 1e-10
                rot  = Rotation.from_rotvec(rng.uniform(-0.3, 0.3) * axis)
                direction = rot.apply(direction)
            else:
                direction = direction + rng.standard_normal(3) * 0.1
                direction /= np.linalg.norm(direction) + 1e-10
            step     = rng.uniform(*Config.STEP_LENGTH_RANGE)
            coords[i] = coords[i-1] + direction * step

    return coords


print('TBM functions loaded.')

In [ ]:
# ── Section 3: Load competition data ─────────────────────────────────────────
t0 = time.time()
print('Loading data...')
train        = pd.read_csv(f'{DATA_PATH}/train_sequences.csv')
val          = pd.read_csv(f'{DATA_PATH}/validation_sequences.csv')
test         = pd.read_csv(f'{DATA_PATH}/test_sequences.csv')
train_labels = pd.read_csv(f'{DATA_PATH}/train_labels.csv', low_memory=False)
val_labels   = pd.read_csv(f'{DATA_PATH}/validation_labels.csv', low_memory=False)
print(f'  Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

print('Building structure lookup...')
t1 = time.time()
train_structs = extract_structures(train_labels)
val_structs   = extract_structures(val_labels)
all_structs   = {**train_structs, **val_structs}
all_seqs      = pd.concat([train, val], ignore_index=True)
print(f'  {len(all_structs)} structures (train+val) in {time.time()-t1:.1f}s')
print(f'Loaded in {time.time()-t0:.1f}s')

In [ ]:
# ── Section 4: TBM Phase — always fills 5 slots, no strict thresholds ────────
# Key design: TBM is guaranteed to produce N predictions for every target.
# Protenix is purely additive — if it fails, TBM predictions are kept as-is.
print('\n── TBM Phase ───────────────────────────────────────────────────────────')

N = Config.NUM_PREDICTIONS
tbm_predictions = {}   # {target_id: [array(L,3) x N]}  always N entries
protenix_queue  = []   # [(target_id, sequence)]  for Protenix enhancement
protenix_slots  = {}   # {target_id: n_slots}  how many slots Protenix may replace
t_tbm = time.time()

for i, row in test.iterrows():
    target_id = row['target_id']
    sequence  = row['sequence']
    cutoff    = row.get('temporal_cutoff', None)

    templates, best_sim = find_templates(
        sequence, all_seqs, all_structs, date_cutoff=cutoff, top_k=N
    )

    preds = []

    # v8 approach: each template gets its own morphed prediction slot
    # This gives structurally diverse predictions vs transforms of one template
    for tmpl_idx, (tid_t, tseq_t, score_t, tcoords_t, a_aln_t, b_aln_t) in enumerate(templates):
        morphed  = morph_template(sequence, a_aln_t, b_aln_t, tcoords_t)
        refined  = refine_geometry(morphed, sequence, reliability=score_t)
        noise_s  = max(Config.NOISE_SCALE_MIN, 0.8 - score_t)
        if score_t > 0.5:
            noise_s *= 0.5
        seed = (int(row.name) * 10_000_000_000 + tmpl_idx * 10_007) % (2**32)
        rng  = np.random.default_rng(seed)
        preds.append(refined + rng.normal(0, noise_s, refined.shape))
        if len(preds) >= N:
            break

    # De-novo fallback for any unfilled slots (refine de-novo, same as v8)
    while len(preds) < N:
        seed  = (int(row.name) * 10_000_000_000 + len(preds) * 10_007) % (2**32)
        rng   = np.random.default_rng(seed)
        dnov  = make_de_novo_structure(sequence, rng=rng)
        dnov  = refine_geometry(dnov, sequence, reliability=Config.DEFAULT_RELIABILITY)
        if preds:
            c0   = np.mean(preds[0], axis=0)
            dnov = c0 + (dnov - np.mean(dnov, axis=0)) * rng.uniform(0.8, 1.2)
        preds.append(dnov)

    tbm_predictions[target_id] = preds  # always exactly N predictions

    # Protenix queue: targets where Protenix can genuinely help
    # (weak/no templates, OR long sequences TBM can't handle)
    n_real_templates = len(templates)
    needs_protenix   = (n_real_templates < N or best_sim < Config.TBM_ROUTING_SIM_THRESHOLD
                        or len(sequence) > Config.MAX_ALIGN_LENGTH)
    if needs_protenix and PROTENIX_AVAILABLE:
        n_prot = N - min(n_real_templates, N) if n_real_templates < N else max(2, N - n_real_templates)
        n_prot = max(n_prot, 2)
        protenix_queue.append((target_id, sequence))
        protenix_slots[target_id] = n_prot

    src = f'tmpl={n_real_templates}  sim={best_sim:.2f}'
    prt = f'  prot={protenix_slots.get(target_id, 0)}' if target_id in protenix_slots else ''
    print(f'  [{i+1:2d}/28] {target_id:8s}  len={len(sequence):5d}  {src}{prt}')

print(f'\nTBM done in {time.time()-t_tbm:.1f}s  |  Protenix queue: {len(protenix_queue)}')

In [ ]:
# ── Section 5: Protenix helper functions ─────────────────────────────────────
import json


def pick_diverse(coords_pool, k):
    """Greedy max-min diversity: pick k most spread-out structures from pool."""
    n = coords_pool.shape[0]
    if n <= k:
        return coords_pool
    chosen, remaining = [0], list(range(1, n))
    while len(chosen) < k and remaining:
        scores = [min(np.mean(np.linalg.norm(coords_pool[r] - coords_pool[c], axis=1))
                      for c in chosen)
                  for r in remaining]
        best = remaining[int(np.argmax(scores))]
        chosen.append(best); remaining.remove(best)
    return coords_pool[chosen]


def _deep_merge(base, override):
    """Deep-merge two dicts so nested keys aren't lost on shallow update."""
    result = dict(base)
    for k, v in override.items():
        if k in result and isinstance(result[k], dict) and isinstance(v, dict):
            result[k] = _deep_merge(result[k], v)
        else:
            result[k] = v
    return result


def build_protenix_configs(input_json_path, dump_dir, model_name, n_sample):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    # model_name must match checkpoint filename: <model_name>.pt in load_checkpoint_dir
    model_cfg = model_configs[model_name]

    # CRITICAL: data_configs MUST be wrapped under "data" key — Protenix model
    # accesses configs.data.ccd_components_file etc. (not as top-level keys).
    # Matches the structure in inference.py:run():
    #   base_configs = {**configs_base, **{"data": data_configs}, **inference_configs}
    base = {**configs_base, **{"data": data_configs}, **inference_configs}
    merged = _deep_merge(base, model_cfg)

    # Pass N_sample via arg_str using dotted notation (supported by parse_configs)
    arg_str = (
        f'--model_name {model_name} '
        f'--input_json_path {input_json_path} '
        f'--dump_dir {dump_dir} '
        f'--use_msa false '
        f'--use_template false '
        f'--use_rna_msa true '
        f'--sample_diffusion.N_sample {n_sample} '
        f'--seeds 42'
    )
    return parse_configs(merged, arg_str=arg_str, fill_required_with_null=True)


def _extract_c1prime(raw_coords, feat, seq_len):
    if 'centre_atom_mask' in feat:
        mask = feat['centre_atom_mask'] == 1
    elif 'center_atom_mask' in feat:
        mask = feat['center_atom_mask'] == 1
    elif 'atom_to_tokatom_idx' in feat:
        m11 = feat['atom_to_tokatom_idx'] == 11
        m12 = feat['atom_to_tokatom_idx'] == 12
        mask = m11 if abs(int(m11.sum()) - seq_len) < abs(int(m12.sum()) - seq_len) else m12
    else:
        raise ValueError('No atom mask in feat')
    coords = raw_coords[:, mask, :].detach().cpu().numpy()
    if np.all(np.linalg.norm(coords[0, 1:] - coords[0, :-1], axis=-1) < 1e-4):
        raise ValueError('Collapsed prediction')
    return coords


def _get_coords(output):
    import torch
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, dict):
        for k in ('coordinate', 'coordinates', 'final_atom_positions', 'pred_positions'):
            if k in output:
                return output[k]
        raise ValueError(f'No coord key in {list(output.keys())}')
    return output


def run_protenix_batch(target_list, n_sample=5):
    """
    Run Protenix inference for a list of (target_id, sequence) pairs.
    Uses InferenceDataset directly (not get_inference_dataloader) to avoid
    DistributedSampler complexity in single-GPU Kaggle environment.
    """
    import torch
    from runner.inference import InferenceRunner, update_gpu_compatible_configs, update_inference_configs
    from protenix.data.inference.infer_dataloader import InferenceDataset

    n_gen = max(6, n_sample)
    records = [{'name': tid, 'covalent_bonds': [],
                'sequences': [{'rnaSequence': {'sequence': seq, 'count': 1}}]}
               for tid, seq in target_list]
    inp  = os.path.join(OUT_PATH, 'protenix_input.json')
    dump = os.path.join(OUT_PATH, 'protenix_dump')
    os.makedirs(dump, exist_ok=True)
    with open(inp, 'w') as f:
        json.dump(records, f)

    configs = build_protenix_configs(inp, dump, MODEL_NAME, n_gen)
    update_gpu_compatible_configs(configs)
    runner = InferenceRunner(configs)

    seq_map = dict(target_list)
    results = {}

    # Use InferenceDataset directly (simpler than get_inference_dataloader,
    # avoids DistributedSampler reordering in single-GPU setting)
    dataset = InferenceDataset(configs)
    for i in range(len(dataset)):
        data, atom_array, error_msg = dataset[i]
        name = data.get('sample_name', '')
        if error_msg:
            print(f'  Protenix data error [{i}] {name}: {error_msg[:200]}')
            continue
        if name not in seq_map:
            continue
        try:
            n_token     = data['N_token'].item()
            new_configs = update_inference_configs(configs, n_token)
            # Allow per-target N_sample adjustment
            new_configs.sample_diffusion.N_sample = n_gen
            runner.update_model_configs(new_configs)
            prediction  = runner.predict(data)
            feat        = data['input_feature_dict']
            raw_coords  = _get_coords(prediction)
            coords      = _extract_c1prime(raw_coords, feat, len(seq_map[name]))
            diverse     = pick_diverse(coords, min(n_sample, coords.shape[0]))
            results[name] = diverse
            print(f'  Protenix OK: {name}  shapes={coords.shape}->{diverse.shape}')
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  Protenix FAIL: {name}  {e}')
        torch.cuda.empty_cache()
    return results


def run_protenix_chunked(target_id, sequence, n_sample=5, chunk_size=512, overlap=128):
    n = len(sequence)
    if n <= chunk_size:
        r = run_protenix_batch([(target_id, sequence)], n_sample)
        return r.get(target_id)
    chunks = []
    pos = 0
    while pos < n:
        end = min(pos + chunk_size, n)
        chunks.append((pos, end))
        if end == n: break
        pos += chunk_size - overlap
    print(f'  Chunking {target_id} ({n} nt) into {len(chunks)} chunks')
    chunk_results = {}
    for ci, (s, e) in enumerate(chunks):
        cid = f'{target_id}_c{ci}'
        r   = run_protenix_batch([(cid, sequence[s:e])], n_sample)
        if cid not in r:
            return None
        chunk_results[ci] = (s, e, r[cid])
    full = np.full((n_sample, n, 3), np.nan, dtype=np.float32)
    s0, e0, c0 = chunk_results[0]
    full[:, s0:e0] = c0
    for ci in range(1, len(chunks)):
        si, ei, ci_coords = chunk_results[ci]
        ps, pe, _         = chunk_results[ci-1]
        ol = pe - si
        for s in range(n_sample):
            ref = full[s, si:pe]
            mov = ci_coords[s, :ol]
            cr, cm = np.mean(ref, 0), np.mean(mov, 0)
            if ol > 3:
                rot, _ = Rotation.align_vectors(ref - cr, mov - cm)
                aligned = rot.apply(ci_coords[s] - cm) + cr
            else:
                aligned = ci_coords[s] + (ref[0] - mov[0])
            full[s, pe:ei] = aligned[ol:]
    for s in range(n_sample):
        for d in range(3):
            col = full[s, :, d]
            nan = np.isnan(col)
            if nan.any() and not nan.all():
                idx = np.arange(n)
                col[nan] = np.interp(idx[nan], idx[~nan], col[~nan])
                full[s, :, d] = col
    return full


print('Protenix functions loaded.')


In [ ]:
# ── Section 6: Protenix Phase (optional enhancement — TBM is always the floor) 
print('\n── Protenix Phase ──────────────────────────────────────────────────────')

# Max sequence length for Protenix. Beyond this, chunking becomes too slow
# (9MME = 4640 nt → 13 chunks × 5 samples = 65 GPU calls; TBM de-novo equally good)
PROTENIX_MAX_LEN = 1000

protenix_predictions = {}  # {target_id: array (n_slots, L, 3)}

if not PROTENIX_AVAILABLE:
    print('Protenix not available — keeping pure TBM predictions.')
elif not torch.cuda.is_available():
    print('No GPU — keeping pure TBM predictions.')
elif not protenix_queue:
    print('No targets queued for Protenix.')
else:
    # Split: ≤512 = single-call batch, 512-PROTENIX_MAX_LEN = chunked, >PROTENIX_MAX_LEN = skip
    short    = [(tid, seq) for tid, seq in protenix_queue if len(seq) <= 512]
    mid_     = [(tid, seq) for tid, seq in protenix_queue if 512 < len(seq) <= PROTENIX_MAX_LEN]
    skipped  = [(tid, seq) for tid, seq in protenix_queue if len(seq) > PROTENIX_MAX_LEN]
    if skipped:
        print(f'Skipping {len(skipped)} very long sequences from Protenix (TBM de-novo used):')
        for tid, seq in skipped:
            print(f'  {tid}: {len(seq)} nt')

    max_prot = max(protenix_slots.values()) if protenix_slots else 5
    n_gen    = max(6, max_prot)

    if short:
        print(f'Running Protenix batch: {len(short)} short targets, n_gen={n_gen}...')
        t_p = time.time()
        try:
            batch = run_protenix_batch(short, n_sample=n_gen)
            for tid, coords in batch.items():
                protenix_predictions[tid] = coords[:protenix_slots.get(tid, n_gen)]
            print(f'  Done in {time.time()-t_p:.1f}s  ({len(batch)}/{len(short)} OK)')
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  Batch failed: {e}  (keeping TBM for all)')

    for tid, seq in mid_:
        n_prot = protenix_slots.get(tid, max_prot)
        print(f'Protenix chunked: {tid} ({len(seq)} nt)...')
        t_p = time.time()
        try:
            coords = run_protenix_chunked(tid, seq, n_sample=max(6, n_prot))
            if coords is not None:
                protenix_predictions[tid] = coords[:n_prot]
                print(f'  {tid} done in {time.time()-t_p:.1f}s')
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  {tid} failed: {e}  (keeping TBM)')

print(f'\nProtenix produced predictions for: {len(protenix_predictions)} targets')
print('Targets without Protenix will use full TBM predictions.')


In [ ]:
# ── Section 7: Build submission.csv ──────────────────────────────────────────
print('\n── Building submission ─────────────────────────────────────────────────')

N = Config.NUM_PREDICTIONS
records = []

for i, row in test.iterrows():
    target_id = row['target_id']
    sequence  = row['sequence']
    L         = len(sequence)

    tbm_preds   = tbm_predictions[target_id]          # always N entries
    prot_arr    = protenix_predictions.get(target_id) # (n_prot, L, 3) or None
    n_prot_have = prot_arr.shape[0] if prot_arr is not None else 0
    n_prot_need = protenix_slots.get(target_id, 0)
    n_prot_use  = min(n_prot_have, n_prot_need)
    n_tbm_use   = N - n_prot_use  # TBM fills all remaining slots

    all_preds = list(tbm_preds[:n_tbm_use])
    for pi in range(n_prot_use):
        all_preds.append(prot_arr[pi])

    # Centre at origin
    all_preds = [p - np.mean(p, axis=0) for p in all_preds[:N]]

    for resid in range(L):
        record = {'ID': f'{target_id}_{resid+1}', 'resname': sequence[resid], 'resid': resid+1}
        for m in range(N):
            x, y, z = all_preds[m][resid]
            record[f'x_{m+1}'] = float(np.clip(x, -999.999, 9999.999))
            record[f'y_{m+1}'] = float(np.clip(y, -999.999, 9999.999))
            record[f'z_{m+1}'] = float(np.clip(z, -999.999, 9999.999))
        records.append(record)

    print(f'  [{i+1:2d}/28] {target_id:8s}  TBM:{n_tbm_use}  Prot:{n_prot_use}')

cols = ['ID', 'resname', 'resid']
for m in range(1, N+1):
    for ax in ('x', 'y', 'z'):
        cols.append(f'{ax}_{m}')

submission = pd.DataFrame(records)[cols]
out_file   = os.path.join(OUT_PATH, 'submission.csv')
submission.to_csv(out_file, index=False)

print(f'\nSaved {len(submission):,} rows -> {out_file}')
print(submission.head(3).to_string())
print('\nDone!')